# Mapillary loader: points, images, 100 m cells, and trajectories

This Colab-style notebook shows how to query Mapillary street-level imagery around Mainz.

**Learning goals**

- Store an API token without committing it to Git.
- Query Mapillary image metadata by a small bounding box.
- Visualize image points and thumbnails on a map.
- Query images for three geospatial access patterns:
  1. a 100 m census-style grid cell,
  2. a point with a distance buffer,
  3. a trajectory with a corridor buffer.

**Important**

Do not paste your token into a notebook cell that will be committed. Use Colab Secrets or an environment variable.


In [ ]:
# Colab setup
!pip -q install geopandas shapely pyproj folium requests python-dotenv

In [ ]:
from pathlib import Path
import math
import os

import folium
import geopandas as gpd
import pandas as pd
import requests
from pyproj import Transformer
from shapely.geometry import LineString, Point, box

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

## 1. API token

Create a Mapillary access token in Meta for Developers. In Colab, store it as a secret named `MAPILLARY_ACCESS_TOKEN`. For local Jupyter, set the same environment variable before starting the notebook.

Local `.env` files are also supported through `data_loaders/mapillary/config/.env`, which is ignored by Git in this repository.


In [ ]:
def load_local_env():
    if load_dotenv is None:
        return
    candidates = [
        Path("../config/.env"),
        Path("data_loaders/mapillary/config/.env"),
    ]
    for candidate in candidates:
        if candidate.exists():
            load_dotenv(candidate)


def get_mapillary_token():
    try:
        from google.colab import userdata
        token = userdata.get("MAPILLARY_ACCESS_TOKEN")
        if token:
            return token
    except Exception:
        pass

    load_local_env()
    token = os.getenv("MAPILLARY_ACCESS_TOKEN")
    if token:
        return token

    raise RuntimeError(
        "No Mapillary token found. Add MAPILLARY_ACCESS_TOKEN to Colab Secrets "
        "or set it as an environment variable."
    )


MAPILLARY_ACCESS_TOKEN = get_mapillary_token()

## 2. Study area and helper functions

The example uses a tiny area near HS Mainz. The coordinate was provided as `lat, lon = 49.9885, 8.22708`; API helpers use the usual WGS84 `lon, lat` order.


In [ ]:
# Tiny demo bounding box around the selected HS Mainz area.
# Input coordinate was given as lat/lon: 49.9885, 8.22708.
# Mapillary expects WGS84 lon/lat order in bbox and geometry helpers.
# Dense Mapillary areas can fail even below the formal 0.010 square-degree limit,
# so the classroom demo starts with a very small area around the selected point.
MAINZ_BBOX_WGS84 = (8.22508, 49.9870, 8.22908, 49.9900)
MAPILLARY_MAX_BBOX_AREA_DEG2 = 0.010

# Example point near Hochschule Mainz area, stored as lon/lat.
SAMPLE_POINT_LONLAT = (8.22708, 49.9885)

# Short example trajectory near the selected point.
SAMPLE_TRAJECTORY_LONLAT = [
    (8.2258, 49.9879),
    (8.22708, 49.9885),
    (8.2284, 49.9891),
]

# Minimal fields keep API responses small and reliable for demos.
MAPILLARY_FIELDS = ",".join([
    "id",
    "computed_geometry",
    "captured_at",
    "thumb_256_url",
])

In [ ]:
def graph_get(endpoint, params=None, token=MAPILLARY_ACCESS_TOKEN):
    params = dict(params or {})
    params["access_token"] = token
    url = f"https://graph.mapillary.com/{endpoint.lstrip('/')}"
    response = requests.get(url, params=params, timeout=60)
    if response.status_code >= 400:
        raise RuntimeError(f"Mapillary API error {response.status_code}: {response.text[:500]}")
    return response.json()


def bbox_area_degrees(bbox_wgs84):
    min_lon, min_lat, max_lon, max_lat = bbox_wgs84
    return (max_lon - min_lon) * (max_lat - min_lat)


def validate_mapillary_bbox(bbox_wgs84, max_area=MAPILLARY_MAX_BBOX_AREA_DEG2):
    area = bbox_area_degrees(bbox_wgs84)
    if area > max_area:
        raise ValueError(
            f"BBox area is {area:.3f} square degrees, above Mapillary's {max_area:.3f} limit. "
            "Use a smaller demo bbox or split the area into tiles."
        )
    return area


def fetch_images_by_bbox(bbox_wgs84, limit=20, fields=MAPILLARY_FIELDS):
    """Fetch image metadata for a bbox: (min_lon, min_lat, max_lon, max_lat)."""
    validate_mapillary_bbox(bbox_wgs84)
    params = {
        "fields": fields,
        "bbox": ",".join(map(str, bbox_wgs84)),
        "limit": min(limit, 100),
    }
    data = graph_get("images", params=params)
    rows = data.get("data", [])
    return rows[:limit]


def empty_images_gdf(crs="EPSG:4326"):
    return gpd.GeoDataFrame(
        columns=["id", "computed_geometry", "captured_at", "thumb_256_url", "lon", "lat", "geometry"],
        geometry="geometry",
        crs=crs,
    )


def images_to_gdf(rows):
    records = []
    for row in rows or []:
        geom = row.get("computed_geometry") or {}
        coords = geom.get("coordinates")
        if not coords or len(coords) < 2:
            continue
        lon, lat = coords[0], coords[1]
        record = dict(row)
        record["lon"] = lon
        record["lat"] = lat
        record["geometry"] = Point(lon, lat)
        records.append(record)

    if not records:
        return empty_images_gdf()
    return gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")


def bbox_from_geometry_wgs84(geom_wgs84):
    minx, miny, maxx, maxy = geom_wgs84.bounds
    return (minx, miny, maxx, maxy)


def buffer_point_wgs84(lon, lat, radius_m):
    point = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs("EPSG:3035")
    buffered = point.buffer(radius_m).to_crs("EPSG:4326").iloc[0]
    return buffered


def buffer_trajectory_wgs84(lonlat_points, radius_m):
    line = LineString(lonlat_points)
    gs = gpd.GeoSeries([line], crs="EPSG:4326").to_crs("EPSG:3035")
    buffered = gs.buffer(radius_m).to_crs("EPSG:4326").iloc[0]
    return buffered


def census_100m_cell_around_point(lon, lat):
    """Create the EPSG:3035 100 m grid cell containing a WGS84 point."""
    to_3035 = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)
    x, y = to_3035.transform(lon, lat)
    cell_minx = math.floor(x / 100) * 100
    cell_miny = math.floor(y / 100) * 100
    cell_3035 = box(cell_minx, cell_miny, cell_minx + 100, cell_miny + 100)
    cell_wgs84 = gpd.GeoSeries([cell_3035], crs="EPSG:3035").to_crs("EPSG:4326").iloc[0]
    grid_id = f"CRS3035RES100mN{int(cell_miny)}E{int(cell_minx)}"
    return grid_id, cell_wgs84

## 3. Query Mapillary images near the selected HS Mainz area

This is the simplest query pattern. The default bbox is intentionally tiny and centered on the selected coordinate `49.9885, 8.22708`, so API responses stay small.


In [ ]:
print(f"Demo bbox area: {bbox_area_degrees(MAINZ_BBOX_WGS84):.6f} square degrees")
rows = fetch_images_by_bbox(MAINZ_BBOX_WGS84, limit=10)
gdf_images = images_to_gdf(rows)
print(f"Fetched {len(gdf_images)} image points near the selected HS Mainz area")
gdf_images.head()

## 4. Visualize image points


In [ ]:
def map_center(gdf=None, fallback_lonlat=SAMPLE_POINT_LONLAT, overlay_geometry=None):
    if gdf is not None and len(gdf):
        center = gdf.geometry.unary_union.centroid
        return [center.y, center.x]
    if overlay_geometry is not None:
        center = overlay_geometry.centroid
        return [center.y, center.x]
    lon, lat = fallback_lonlat
    return [lat, lon]


def map_images(gdf, overlay_geometry=None, overlay_name="query area", zoom_start=17):
    if gdf is None:
        gdf = empty_images_gdf()

    m = folium.Map(location=map_center(gdf, overlay_geometry=overlay_geometry), zoom_start=zoom_start, tiles="cartodbpositron")

    if overlay_geometry is not None:
        folium.GeoJson(
            overlay_geometry.__geo_interface__,
            name=overlay_name,
            style_function=lambda feature: {
                "color": "#1f77b4",
                "weight": 2,
                "fillOpacity": 0.08,
            },
        ).add_to(m)

    if len(gdf):
        for _, row in gdf.iterrows():
            thumb = row.get("thumb_256_url", "")
            image_id = row.get("id", "")
            captured = row.get("captured_at", "")
            popup = f"<b>{image_id}</b><br>{captured}"
            if thumb:
                popup += f"<br><img src='{thumb}' width='180'>"
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=4,
                color="#d62728",
                fill=True,
                fill_opacity=0.8,
                popup=folium.Popup(popup, max_width=220),
            ).add_to(m)
    else:
        folium.Marker(
            location=map_center(gdf, overlay_geometry=overlay_geometry),
            tooltip="No Mapillary images returned for this small query area",
        ).add_to(m)

    folium.LayerControl().add_to(m)
    return m


map_images(gdf_images, overlay_geometry=box(*MAINZ_BBOX_WGS84), overlay_name="demo bbox", zoom_start=17)

## 5. Query by 100 m census-style grid cell


In [ ]:
grid_id, cell_geom = census_100m_cell_around_point(*SAMPLE_POINT_LONLAT)
print(grid_id)

cell_rows = fetch_images_by_bbox(bbox_from_geometry_wgs84(cell_geom), limit=10)
gdf_cell = images_to_gdf(cell_rows)

# The API query uses a bbox; filter precisely to the cell polygon.
if len(gdf_cell):
    gdf_cell = gdf_cell[gdf_cell.within(cell_geom)].copy()

print(f"Images inside cell: {len(gdf_cell)}")
map_images(gdf_cell, overlay_geometry=cell_geom, overlay_name=grid_id, zoom_start=17)

## 6. Query by point buffer


In [ ]:
point_buffer = buffer_point_wgs84(*SAMPLE_POINT_LONLAT, radius_m=80)
point_rows = fetch_images_by_bbox(bbox_from_geometry_wgs84(point_buffer), limit=10)
gdf_point = images_to_gdf(point_rows)

if len(gdf_point):
    gdf_point = gdf_point[gdf_point.within(point_buffer)].copy()

print(f"Images within point buffer: {len(gdf_point)}")
map_images(gdf_point, overlay_geometry=point_buffer, overlay_name="80 m point buffer", zoom_start=17)